# Haiku MIL Classification

**Project Name:** Haiku (renamed from Haiku)

## Purpose
- Publication-ready notebook for reproducible training/evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys; sys.path.insert(0, "../src")
from haiku import setup_notebook, seed_everything
setup_notebook(project_root="..")
seed_everything(42)

In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from transformers import BertTokenizer
import pandas as pd
import numpy as np
import torch.nn.functional as F
import json
import sys


#target_patches_id = np.load('/home/yancui/Haiku/checkpoints/Trimodal_20250905-2158_full_trainset/holdout_patch_id.npy')

codex_embedding = torch.load('/home/yancui/Haiku/res_embedding_303/new_codex_embedding.pt')

virtual_codex_embedding = torch.load('/home/yancui/Haiku/res_embedding_303/new_virtual_codex_embedding.pt')

region_label = torch.load('/home/yancui/Haiku/res_embedding_303/new_region_label.pt')

print(codex_embedding.shape)

sample_ids = list(json.load(open('/home/yancui/Haiku/overlap_samples_new.json')).keys())

ref_ids = sorted(sample_ids)

# Build bags of embeddings for each patient
# The mapping from region_label[i] -> ref_ids[region_label[i]] will give you a region/acquisition ID
patient_he_bags = dict()        # patient_id: [he_embedding_tensors]
patient_codex_bags = dict()  # patient_id: [codex_embedding_tensors]
patient_virt_bags = dict()   # patient_id: [virtual_codex_embedding_tensors]
patient_musk_bags = dict()   # patient_id: [musk_embedding_tensors]
patient_labels = dict()
patient_concat_bags = dict()    # patient_id: label (list or scalar per patient, depending)

for idx, region_l in enumerate(region_label):
    # Lookup the patient_id for this region/acquisition

    region_id = ref_ids[region_l]

    # Group embeddings by patient
    #patient_he_bags.setdefault(region_id, []).append(he_embedding[idx])
    patient_codex_bags.setdefault(region_id, []).append(codex_embedding[idx])
    patient_virt_bags.setdefault(region_id, []).append(virtual_codex_embedding[idx])
    #patient_musk_bags.setdefault(region_id, []).append(musk_embedding[idx])
    #patient_concat_bags.setdefault(region_id, []).append(torch.cat([he_embedding[idx], codex_embedding[idx]], dim=0))

    # Optionally: Also maintain region-labels or per-patient labels
    # Here, defaulting to using the region's integer label.
    patient_labels.setdefault(region_id, []).append(region_label[idx])

# Optionally, if you want a "single label per patient" for classification (e.g. majority, first, or a function):
# Example (majority label per patient):
from collections import Counter
patient_majority_labels = {
    pid: Counter(lbls).most_common(1)[0][0]
    for pid, lbls in patient_labels.items()
}

# patient_bags, patient_codex_bags, patient d_virt_bags, patient_musk_bags now each map a patient_id -> list of embeddings as bags
# patient_labels: patient_id -> list of integer region label per patch, or use patient_majority_labels for (patient_id -> majority label)


In [ ]:
import os
import pandas as pd

csv_list = ['/home/yancui/Haiku/Lymphoma_response-to-RCHOP (1).csv',
'/home/yancui/Haiku/Melanoma_response-to-immunotherapy (1).csv',
'/home/yancui/Haiku/CRC_terminal_survival.csv']


need_new_acq_ids = []

df_list = []

for i in csv_list:
    df = pd.read_csv(i)
    new_acq_ids = df['ACQUISITION_ID'].tolist()
    df_list.append(df)



In [ ]:
df_list[1].columns

In [ ]:
df_list[2]['description'].unique()

In [ ]:
df_list[2]['description'].shape, df_list[1]['description'].shape


In [ ]:
df_list[1]['description'].unique()

In [ ]:
survial_length_dict = {}
survial_status_dict = { }
response_dict = {}
treatment_dict = {}

In [ ]:
response_dict['0'] = 'Response-binary'
treatment_dict['0'] = 'treatment'

In [ ]:
survial_length_dict['1'] = 'survival'
survial_status_dict['1'] = 'survival_status'
response_dict['1'] = 'Response-binary'
treatment_dict['1'] = 'treatment'

In [ ]:
survial_length_dict['2'] = 'FOLLOW UP (months)'
survial_status_dict['2'] = 'survival_status'
response_dict['2'] = 'Outcome-binary'
treatment_dict['2'] = 'treatment'

In [ ]:
# End-to-end MIL Classification (Binary): 5-fold CV with AUROC/AUPRC curves + boxplots
# Expected objects already in scope (same as your script):
#   - patient_virt_bags    : {bag_id: [np.ndarray(Din,), ...]}  # Baseline
#   - patient_codex_bags   : {bag_id: [np.ndarray(Din,), ...]}  # Ours
#   - df_list, sample_ids, response_dict, treatment_dict
#
# Notes:
# - Figures are saved as SVG (editable text in Illustrator) and PNG.
# - Results JSON contains per-fold metrics and pooled metrics.
# - Uses StratifiedKFold on bag IDs.

import os, json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, auc,
    accuracy_score, f1_score
)

# ----------------------------
# 0) Reproducibility + Device
# ----------------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Keep text editable in SVG exports
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# ---------------------------------------
# 1) Dataset and collate for padded bags
# ---------------------------------------
class MILDataset(Dataset):
    def __init__(self, emb_dict: Dict[str, List[np.ndarray]],
                 labels: Dict[str, int],
                 bag_ids: List[str]):
        self.ids, self.bags, self.y = [], [], []
        for bid in bag_ids:
            if (bid not in emb_dict) or (bid not in labels):
                continue
            insts = emb_dict[bid]
            if len(insts) == 0:
                continue
            X = np.stack(insts, axis=0).astype(np.float32)  # [N,D]
            self.ids.append(bid)
            self.bags.append(torch.from_numpy(X))
            self.y.append(int(labels[bid]))
        if len(self.bags) == 0:
            raise ValueError("Empty MILDataset—check dictionaries/split.")
        self.Din = self.bags[0].shape[1]

    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return self.bags[i], torch.tensor(self.y[i], dtype=torch.long), self.ids[i]

def pad_collate(batch):
    bags, ys, ids = zip(*batch)
    B = len(bags)
    Nmax = max(b.shape[0] for b in bags)
    D = bags[0].shape[1]
    X = torch.zeros(B, Nmax, D, dtype=torch.float32)
    M = torch.zeros(B, Nmax, dtype=torch.bool)
    for i, b in enumerate(bags):
        n = b.shape[0]
        X[i, :n] = b
        M[i, :n] = True
    y = torch.stack(ys)  # [B]
    return X, M, y, list(ids)

# --------------------------------
# 2) MIL Model (Encoder + Pool + Head)
# --------------------------------
class AttnPool(nn.Module):
    def __init__(self, d, hidden=128):
        super().__init__()
        self.V = nn.Linear(d, hidden)
        self.w = nn.Linear(hidden, 1, bias=False)
    def forward(self, H, mask):
        A = self.w(torch.tanh(self.V(H))).squeeze(-1)        # [B,N]
        A = A.masked_fill(~mask, float("-inf"))
        A = torch.softmax(A, dim=1)
        Z = torch.einsum("bn,bnd->bd", A, H)                 # [B,d]
        return Z, A

class MILClassifier(nn.Module):
    def __init__(self, in_dim, embed_dim=128, pool="attn", dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(64),
            nn.Linear(64, embed_dim), nn.ReLU(),
        )
        self.pool_type = pool
        self.pool = AttnPool(embed_dim) if pool == "attn" else None
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1)
        )

    def forward(self, X, mask):
        H = self.encoder(X)  # [B,N,E]
        if self.pool_type == "attn":
            Z, A = self.pool(H, mask)
        elif self.pool_type == "mean":
            Z = (H * mask.unsqueeze(-1)).sum(1) / (mask.sum(1, keepdim=True).clamp_min(1)).to(H.dtype)
            A = None
        elif self.pool_type == "max":
            Hm = H.masked_fill(~mask.unsqueeze(-1), torch.finfo(H.dtype).min/2)
            Z = Hm.max(dim=1).values
            A = None
        else:
            raise ValueError("pool must be one of {'attn','mean','max'}")
        logits = self.head(Z).squeeze(-1)  # [B]
        return logits, A

# --------------------------------
# 3) Train / evaluate routines
# --------------------------------
@torch.no_grad()
def predict_model(model, loader, device):
    model.eval()
    all_ids, all_y, all_logits = [], [], []
    for X, M, y, ids in loader:
        X, M = X.to(device), M.to(device)
        logits, _ = model(X, M)
        all_ids += ids
        all_y += y.numpy().tolist()
        all_logits += logits.cpu().numpy().tolist()
    y_true = np.array(all_y)
    logits = np.array(all_logits)
    probs = 1/(1+np.exp(-logits))
    return all_ids, y_true, probs, logits

def evaluate_binary(y_true, prob, threshold=0.5):
    if not isinstance(y_true, np.ndarray): y_true = np.array(y_true)
    if not isinstance(prob, np.ndarray): prob = np.array(prob)
    auroc = roc_auc_score(y_true, prob) if len(np.unique(y_true))>1 else np.nan
    auprc = average_precision_score(y_true, prob)
    y_pred = (prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    fpr, tpr, _ = roc_curve(y_true, prob) if len(np.unique(y_true))>1 else (np.array([0,1]), np.array([0,1]), None)
    prec, rec, _ = precision_recall_curve(y_true, prob)
    return {
        "auroc": float(auroc), "auprc": float(auprc), "acc": float(acc), "f1_macro": float(f1m),
        "fpr": fpr.tolist(), "tpr": tpr.tolist(), "prec": prec.tolist(), "rec": rec.tolist()
    }

def train_mil_classifier(emb_dict, labels, train_ids, val_ids,
                         pool="attn", batch_size=128, max_epochs=30, lr=1e-3, wd=1e-4, device="cpu"):
    set_seed(42)
    train_ds = MILDataset(emb_dict, labels, train_ids)
    val_ds   = MILDataset(emb_dict, labels, val_ids)
    in_dim = train_ds.Din

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=pad_collate)

    model = MILClassifier(in_dim=in_dim, embed_dim=128, pool=pool, dropout=0.2).to(device)

    # class imbalance handling using TRAIN fold only
    y_train = np.array([labels[_id] for _id in train_ds.ids])
    pos = (y_train==1).sum(); neg = (y_train==0).sum()
    pos_weight = torch.tensor([neg / max(pos,1)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    for epoch in range(1, max_epochs+1):
        model.train()
        running = 0.0; n = 0
        for X, M, y, _ in train_loader:
            X, M, y = X.to(device), M.to(device), y.to(device).float()
            logits, _ = model(X, M)
            loss = criterion(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * X.size(0); n += X.size(0)

        # Light val log
        _, yv, pv, _ = predict_model(model, val_loader, device)
        val_metrics = evaluate_binary(yv, pv)
        print(f"[Epoch {epoch:03d}] train_loss={running/max(n,1):.4f} "
              f"val_AUPRC={val_metrics['auprc']:.4f} val_AUROC={val_metrics['auroc']:.4f} "
              f"val_F1m={val_metrics['f1_macro']:.4f} val_ACC={val_metrics['acc']:.4f}")

    return model

# ---------------------------------------
# 4) Utilities
# ---------------------------------------
def _to_scalar_label(v):
    """Coerce df cell -> int(0/1). Handles 1-length arrays."""
    arr = np.asarray(v)
    if arr.shape == ():
        return int(arr)
    if arr.size == 1:
        return int(arr.reshape(-1)[0])
    # If multi-valued, take first element (adjust if needed)
    return int(arr.reshape(-1)[0])

def make_loader(emb_dict, labels, ids, batch=256):
    ds = MILDataset(emb_dict, labels, ids)
    return DataLoader(ds, batch_size=batch, shuffle=False, collate_fn=pad_collate)

def interpolate_mean_curve(xs, ys_list, grid):
    # xs: list of x-arrays; ys_list: list of y-arrays (same length list as xs)
    # returns mean and std over interpolation to 'grid'
    mats = []
    for x, y in zip(xs, ys_list):
        x = np.asarray(x); y = np.asarray(y)
        # ensure monotonic x for interp
        order = np.argsort(x)
        x = x[order]; y = y[order]
        mats.append(np.interp(grid, x, y))
    mat = np.vstack(mats)
    return mat.mean(0), mat.std(0)

# ---------------------------------------
# 5) Five-fold CV driver (Baseline vs Ours)
# ---------------------------------------
def run_mil_cls_cv5(baseline_embeddings, our_embeddings, labels_dict,
                    out_dir="cv5_outputs",
                    pool="attn", max_epochs=40, batch_size=256, lr=5e-4,
                    device=device, seed=42):
    os.makedirs(out_dir, exist_ok=True)
    set_seed(seed)

    # Bag universe and y for stratification
    all_ids = sorted(set(labels_dict.keys()) & set(baseline_embeddings.keys()) & set(our_embeddings.keys()))
    y_all   = np.array([_to_scalar_label(labels_dict[i]) for i in all_ids], dtype=int)
    print(f"[CV] usable bags = {len(all_ids)}; positives={y_all.sum()} negatives={(y_all==0).sum()}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    # Storage
    per_fold_metrics = {"baseline": [], "ours": []}
    # For pooled curves (concat predictions across folds)
    base_probs_all, base_true_all = [], []
    ours_probs_all, ours_true_all = [], []
    # For mean±std bands (store each fold curve)
    roc_fprs_b, roc_tprs_b = [], []
    roc_fprs_o, roc_tprs_o = [], []
    pr_recs_b, pr_precs_b  = [], []
    pr_recs_o, pr_precs_o  = [], []

    fold_idx = 0
    for tr_idx, te_idx in skf.split(all_ids, y_all):
        fold_idx += 1
        print(f"\n=== Fold {fold_idx}/5 ===")
        train_ids = [all_ids[i] for i in tr_idx]
        test_ids  = [all_ids[i] for i in te_idx]
        # Use small validation set from train for logging (10% of train, stratified)
        # If too small, fall back to using test as validation for logging only
        y_tr = np.array([labels_dict[i] for i in train_ids], dtype=int)
        if len(train_ids) >= 10 and len(np.unique(y_tr)) == 2:
            # 10% split
            from sklearn.model_selection import train_test_split
            tr_ids, va_ids = train_test_split(
                train_ids, test_size=max(1, int(0.1*len(train_ids))),
                random_state=seed, stratify=y_tr
            )
        else:
            tr_ids, va_ids = train_ids, test_ids

        # Train baseline & ours
        print("[Fold] Training Baseline...")
        base_model = train_mil_classifier(
            baseline_embeddings, labels_dict, tr_ids, va_ids,
            pool=pool, batch_size=batch_size, max_epochs=max_epochs, lr=lr, device=device
        )
        print("[Fold] Training Ours...")
        ours_model = train_mil_classifier(
            our_embeddings, labels_dict, tr_ids, va_ids,
            pool=pool, batch_size=batch_size, max_epochs=max_epochs, lr=lr, device=device
        )

        # Evaluate on test fold
        base_loader = make_loader(baseline_embeddings, labels_dict, test_ids, batch_size)
        ours_loader = make_loader(our_embeddings,    labels_dict, test_ids, batch_size)


        _, yt_b, pt_b, _ = predict_model(base_model, base_loader, device)
        _, yt_o, pt_o, _ = predict_model(ours_model, ours_loader, device)

        print(yt_b.shape, pt_b.shape)


        m_b = evaluate_binary(yt_b, pt_b)
        m_o = evaluate_binary(yt_o, pt_o)

        per_fold_metrics["baseline"].append({"fold": fold_idx, **m_b})
        per_fold_metrics["ours"].append({"fold": fold_idx, **m_o})

        # accumulate pooled preds
        base_probs_all.append(pt_b); base_true_all.append(yt_b)
        ours_probs_all.append(pt_o); ours_true_all.append(yt_o)

        # store curves for bands
        roc_fprs_b.append(np.array(m_b["fpr"]))
        roc_tprs_b.append(np.array(m_b["tpr"]))
        roc_fprs_o.append(np.array(m_o["fpr"]))
        roc_tprs_o.append(np.array(m_o["tpr"]))
        pr_recs_b.append(np.array(m_b["rec"]))
        pr_precs_b.append(np.array(m_b["prec"]))
        pr_recs_o.append(np.array(m_o["rec"]))
        pr_precs_o.append(np.array(m_o["prec"]))

    # Pooled metrics (concat across folds)
    base_probs_all = np.concatenate(base_probs_all); base_true_all = np.concatenate(base_true_all)
    ours_probs_all = np.concatenate(ours_probs_all); ours_true_all = np.concatenate(ours_true_all)
    pooled_base = evaluate_binary(base_true_all, base_probs_all)
    pooled_ours = evaluate_binary(ours_true_all, ours_probs_all)

    results = {
        "fold_metrics": per_fold_metrics,
        "pooled": {"baseline": pooled_base, "ours": pooled_ours}
    }

    # Save JSON
    json_path = os.path.join(out_dir, "cv5_results.json")
    with open(json_path, "w") as f:
    print(f"[CV] Saved metrics JSON → {json_path}")

    # ----------------- Plots -----------------
    # 1) ROC curves (per-fold lines + mean±std + pooled bold)
    mean_fpr = np.linspace(0, 1, 400)
    mean_tpr_b, std_tpr_b = interpolate_mean_curve(roc_fprs_b, roc_tprs_b, mean_fpr)
    mean_tpr_o, std_tpr_o = interpolate_mean_curve(roc_fprs_o, roc_tprs_o, mean_fpr)

    plt.figure(figsize=(6.2,5.2))
    # faint per-fold
    for fpr, tpr in zip(roc_fprs_b, roc_tprs_b): plt.plot(fpr, tpr, alpha=0.25, lw=1)
    for fpr, tpr in zip(roc_fprs_o, roc_tprs_o): plt.plot(fpr, tpr, alpha=0.25, lw=1)
    # mean bands
    plt.fill_between(mean_fpr, np.clip(mean_tpr_b-std_tpr_b,0,1), np.clip(mean_tpr_b+std_tpr_b,0,1), alpha=0.15, label="Baseline (±1σ)")
    plt.fill_between(mean_fpr, np.clip(mean_tpr_o-std_tpr_o,0,1), np.clip(mean_tpr_o+std_tpr_o,0,1), alpha=0.15, label="Ours (±1σ)")
    # pooled bold
    fpr_b, tpr_b = np.array(pooled_base["fpr"]), np.array(pooled_base["tpr"])
    fpr_o, tpr_o = np.array(pooled_ours["fpr"]), np.array(pooled_ours["tpr"])
    plt.plot(fpr_b, tpr_b, lw=2.5, label=f"Baseline pooled (AUROC={pooled_base['auroc']:.3f})")
    plt.plot(fpr_o, tpr_o, lw=2.5, label=f"Ours pooled (AUROC={pooled_ours['auroc']:.3f})")
    plt.plot([0,1],[0,1],'--',alpha=0.4)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC — 5-fold CV")
    plt.legend(framealpha=0.9); plt.grid(alpha=0.2); plt.tight_layout()
    roc_svg = os.path.join(out_dir, "roc_cv.svg"); roc_png = os.path.join(out_dir, "roc_cv.png")

    # 2) PR curves (per-fold lines + mean±std + pooled bold + no-skill)
    mean_rec = np.linspace(0, 1, 400)
    mean_prec_b, std_prec_b = interpolate_mean_curve(pr_recs_b, pr_precs_b, mean_rec)
    mean_prec_o, std_prec_o = interpolate_mean_curve(pr_recs_o, pr_precs_o, mean_rec)
    prev = base_true_all.mean() if base_true_all.size>0 else 0.5

    plt.figure(figsize=(6.2,5.2))
    for rc, pr in zip(pr_recs_b, pr_precs_b): plt.plot(rc, pr, alpha=0.25, lw=1)
    for rc, pr in zip(pr_recs_o, pr_precs_o): plt.plot(rc, pr, alpha=0.25, lw=1)
    plt.fill_between(mean_rec, np.clip(mean_prec_b-std_prec_b,0,1), np.clip(mean_prec_b+std_prec_b,0,1), alpha=0.15, label="Baseline (±1σ)")
    plt.fill_between(mean_rec, np.clip(mean_prec_o-std_prec_o,0,1), np.clip(mean_prec_o+std_prec_o,0,1), alpha=0.15, label="Ours (±1σ)")
    rc_b, pr_b = np.array(pooled_base["rec"]), np.array(pooled_base["prec"])
    rc_o, pr_o = np.array(pooled_ours["rec"]), np.array(pooled_ours["prec"])
    plt.plot(rc_b, pr_b, lw=2.5, label=f"Baseline pooled (AUPRC={pooled_base['auprc']:.3f})")
    plt.plot(rc_o, pr_o, lw=2.5, label=f"Ours pooled (AUPRC={pooled_ours['auprc']:.3f})")
    plt.hlines(prev, 0, 1, linestyles="--", alpha=0.4, label=f"No-skill (p={prev:.2f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision–Recall — 5-fold CV")
    plt.legend(framealpha=0.9); plt.grid(alpha=0.2); plt.tight_layout()
    pr_svg = os.path.join(out_dir, "pr_cv.svg"); pr_png = os.path.join(out_dir, "pr_cv.png")

    # 3) Boxplots for AUROC/AUPRC across folds
    aurocs_b = [m["auroc"] for m in per_fold_metrics["baseline"]]
    aurocs_o = [m["auroc"] for m in per_fold_metrics["ours"]]
    auprcs_b = [m["auprc"] for m in per_fold_metrics["baseline"]]
    auprcs_o = [m["auprc"] for m in per_fold_metrics["ours"]]

    fig, axes = plt.subplots(1, 2, figsize=(9.5,4.2), sharey=False)
    axes[0].boxplot([aurocs_b, aurocs_o], labels=["Baseline","Ours"], showmeans=True)
    axes[0].set_title("AUROC (5-fold)")
    axes[0].set_ylim(0,1)
    axes[1].boxplot([auprcs_b, auprcs_o], labels=["Baseline","Ours"], showmeans=True)
    axes[1].set_title("AUPRC (5-fold)")
    axes[1].set_ylim(0,1)
    plt.suptitle("MIL Classification — 5-fold CV boxplots")
    plt.tight_layout()
    box_svg = os.path.join(out_dir, "boxplots_cv.svg"); box_png = os.path.join(out_dir, "boxplots_cv.png")
    print(f"[CV] Saved figures to {out_dir}")

    return results, {"roc_svg": roc_svg, "pr_svg": pr_svg, "box_svg": box_svg}, base_probs_all, ours_probs_all

# ---------------------------------------
# 6) Building labels from df_list and running CV
# ---------------------------------------
def build_labels_from_df(df, sample_ids, response_key, treatment_key):
    region_labels_response = {}
    region_labels_diagnosis = {}  # unused in CV here, but kept for parity
    for region_id in sample_ids:
        if region_id in df['ACQUISITION_ID'].values:
            response_val = df.loc[df['ACQUISITION_ID']==region_id, response_key].values
            treat_val    = df.loc[df['ACQUISITION_ID']==region_id, treatment_key].values
            region_labels_response[region_id]  = _to_scalar_label(response_val)
            region_labels_diagnosis[region_id] = treat_val
    return region_labels_response, region_labels_diagnosis

# -------- Main loop over your df_list (if you have multiple tasks/labels) --------
OUT_DIR_ROOT = "cv5_outputs_all"
os.makedirs(OUT_DIR_ROOT, exist_ok=True)

probs_all_res = []
ours_probs_all_res = []

for i, df in enumerate(df_list[1:]):  # keep your original indexing
    response_key  = response_dict[str(i+1)]
    treatment_key = treatment_dict[str(i+1)]
    labels_response, labels_diagnosis = build_labels_from_df(df, sample_ids, response_key, treatment_key)

    task_dir = os.path.join(OUT_DIR_ROOT, f"task_{i+1}")
    os.makedirs(task_dir, exist_ok=True)
    print(f"\n######## Task {i+1}: response='{response_key}' ########")

    results, fig_paths, base_probs_all, ours_probs_all = run_mil_cls_cv5(
        baseline_embeddings=patient_virt_bags,
        our_embeddings=patient_codex_bags,
        labels_dict=labels_response,
        out_dir=task_dir,
        pool="attn",
        max_epochs=20,
        batch_size=256,
        lr=1e-4,
        device=device,
        seed=42
    )

    probs_all_res.append(base_probs_all)
    ours_probs_all_res.append(ours_probs_all)

    # Optionally: write a tiny index file per task
    with open(os.path.join(task_dir, "README.txt"), "w") as f:
        f.write("Saved files:\n")
        f.write(f"- metrics JSON: {os.path.join(task_dir,'cv5_results.json')}\n")
        f.write(f"- ROC: {fig_paths['roc_svg']}\n- PR: {fig_paths['pr_svg']}\n- Boxplots: {fig_paths['box_svg']}\n")


In [ ]:
import os, json, numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.stats import ranksums
import seaborn as sns


# Set font to Arial, large bold fonts for clarity, and ensure Illustrator compatibility/editability
plt.rcParams['font.family'] = 'Arial'    # use Arial font
plt.rcParams['svg.fonttype'] = 'none'    # don't convert text to path
plt.rcParams['pdf.fonttype'] = 42        # embed as TrueType for AI compatibility
plt.rcParams['ytick.labelsize'] = 10     # Increase y-tick font size
plt.rcParams['font.weight'] = 'bold'     # make text bold
plt.rcParams['axes.labelweight'] = 'bold'# bold axis labels
plt.rcParams['axes.titleweight'] = 'bold'# bold title

def load_cv_results(json_path):
    with open(json_path, "r") as f:
        return json.load(f)

def print_fold_means(results):
    folds_b = results["fold_metrics"]["baseline"]
    folds_o = results["fold_metrics"]["ours"]

    metrics = ["auroc", "auprc"]  # Only show AUROC and AUPRC
    methods = [
        ("Baseline", folds_b),
        ("Ours", folds_o)
    ]
    print("Mean and std (5-fold) for each metric/method:")
    for method_name, fold_list in methods:
        print(f"Method: {method_name}")
        for metric in metrics:
            vals = np.array([f.get(metric, np.nan) for f in fold_list], dtype=float)
            vals = vals[~np.isnan(vals)]
            mean = np.mean(vals) if len(vals) > 0 else np.nan
            std = np.std(vals, ddof=1) if len(vals) > 1 else np.nan
            print(f"  {metric}: {mean:.4f} ± {std:.4f} (n={len(vals)})")
        print()

def print_ranksum_tests(results):
    folds_b = results["fold_metrics"]["baseline"]
    folds_o = results["fold_metrics"]["ours"]
    metrics = ["auroc", "auprc"]  # Only show AUROC and AUPRC
    print("One-sided Wilcoxon rank-sum test (H1: Ours > Baseline) per metric:")
    for metric in metrics:
        vals_b = np.array([f.get(metric, np.nan) for f in folds_b], dtype=float)
        vals_o = np.array([f.get(metric, np.nan) for f in folds_o], dtype=float)
        vals_b = vals_b[~np.isnan(vals_b)]
        vals_o = vals_o[~np.isnan(vals_o)]
        if len(vals_b) > 0 and len(vals_o) > 0:
            try:
                stat, p = ranksums(vals_o, vals_b, alternative='greater')
                print(f"  {metric}: statistic={stat:.3f}, one-sided p-value={p:.4f}")
            except Exception as e:
                print(f"  {metric}: Rank-sum test failed ({e})")
        else:
            print(f"  {metric}: Not enough data for test (len_b={len(vals_b)}, len_o={len(vals_o)})")
    print()


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

def plot_barplot_metrics(results, out_dir):
    '''
    Create a bar plot (with error bars = std over folds) only for AUROC and AUPRC, per method.
    Updated to control y-tick font size and axis line widths.
    '''
    folds_b = results["fold_metrics"]["baseline"]
    folds_o = results["fold_metrics"]["ours"]

    metrics = ["auroc", "auprc"]  # Only AUROC and AUPRC
    metric_names = {
        "auroc": "AUROC",
        "auprc": "AUPRC"
    }

    data = []
    for metric in metrics:
        vals_b = np.array([f.get(metric, np.nan) for f in folds_b], dtype=float)
        vals_o = np.array([f.get(metric, np.nan) for f in folds_o], dtype=float)
        vals_b = vals_b[~np.isnan(vals_b)]
        vals_o = vals_o[~np.isnan(vals_o)]
        mean_b, std_b, n_b = np.nan, np.nan, 0
        mean_o, std_o, n_o = np.nan, np.nan, 0
        if len(vals_b) > 0:
            mean_b, std_b, n_b = np.mean(vals_b), np.std(vals_b, ddof=1), len(vals_b)
        if len(vals_o) > 0:
            mean_o, std_o, n_o = np.mean(vals_o), np.std(vals_o, ddof=1), len(vals_o)
        data.append(dict(Metric=metric_names[metric], Method="Baseline", Mean=mean_b, Std=std_b, N=n_b))
        data.append(dict(Metric=metric_names[metric], Method="Ours", Mean=mean_o, Std=std_o, N=n_o))

    df_plot = pd.DataFrame(data)

    pal = {
        "Baseline": "#54A0DD",
        "Ours": "#E57067"
    }

    # Reorder for plot
    order_metrics = [metric_names[m] for m in metrics]
    order_methods = ["Baseline", "Ours"]

    fig, ax = plt.subplots(figsize=(15, 8))

    width = 0.33
    x = np.arange(len(order_metrics))

    # --- CONFIGURATION VARIABLES ---
    axis_line_width = 1.5   # Set the width of the axis lines (spines)
    tick_font_size = 16     # Set the font size for y-ticks

    for i, method in enumerate(order_methods):
        means = []
        stds = []
        for m in order_metrics:
            row = df_plot[(df_plot.Metric == m) & (df_plot.Method == method)]
            means.append(row["Mean"].values[0])
            stds.append(row["Std"].values[0])

        # Shift for bar width
        x_shift = x + (i-0.5)*width

        # Added error_kw to control error bar line width as well
        ax.bar(x_shift, means, yerr=stds, width=width, capsize=7, label=method,
               color=pal[method], edgecolor="k", linewidth=1.6,
               error_kw={'linewidth': axis_line_width}, # Matches axis line width
               alpha=0.92, zorder=9)

        # Optionally: add value labels
        for xi, mu, sigma in zip(x_shift, means, stds):
            ax.text(xi, mu + sigma + 0.025, f"{mu:.3f}", ha="center", va="bottom", fontsize=11,
                    fontweight="bold", color=pal[method], zorder=12)

    ax.set_xticks(x)
    ax.set_xticklabels(order_metrics, fontsize=14, fontweight="bold")

    # --- NEW: Set Y-tick font size and width ---
    ax.tick_params(axis='y', labelsize=tick_font_size, width=axis_line_width)
    ax.tick_params(axis='x', width=axis_line_width) # Apply width to x-ticks too for consistency

    ax.set_ylabel("CV Mean ± std", fontsize=14)
    ax.set_ylim(0, 1.08)
    ax.legend(fontsize=13, frameon=False)

    # --- NEW: Set Spine (Frame) Line Width ---
    for spine in ax.spines.values():
        spine.set_linewidth(axis_line_width)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.title("MIL Metrics (CV)", fontsize=15, fontweight="bold")
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plt.show()
    plt.close(fig)

def plot_roc_pr_per_fold(results, out_dir):
    '''
    As before: per-fold ROC/PRC curves with points for baseline/ours.
    '''
    folds_b = results["fold_metrics"]["baseline"]
    folds_o = results["fold_metrics"]["ours"]

    os.makedirs(out_dir, exist_ok=True)
    n_folds = max(len(folds_b), len(folds_o))

    for fidx in range(n_folds):
        # ROC
        fig = plt.figure(figsize=(8, 5))
        ax = fig.gca()
        for method, fold_list, color, marker in zip(
            ["Baseline", "Ours"],
            [folds_b, folds_o],
            ["#54A0DD", "#E57067"],
            ['o', '^']
        ):
            if fidx >= len(fold_list):
                continue
            fold = fold_list[fidx]
            fpr = np.array(fold.get("fpr", []))
            tpr = np.array(fold.get("tpr", []))
            auroc = fold.get("auroc", float("nan"))
            if fpr.size > 1:
                plt.plot(
                    fpr, tpr, label=f"{method} (AUROC={auroc:.3f})", lw=2, color=color
                )
                plt.scatter(fpr, tpr, color=color, marker=marker, s=33, edgecolor="black", alpha=0.8, zorder=10, label=None)
        plt.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.45)
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve — Fold {fidx + 1}")
        handles, labels = ax.get_legend_handles_labels()
        # Deduplicate labels
        out_handles = []; out_labels = []; seen = set()
        for h, l in zip(handles, labels):
            if l not in seen:
                seen.add(l)
                out_handles.append(h)
                out_labels.append(l)
        plt.legend(out_handles, out_labels)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        out_path_svg = os.path.join(out_dir, f"roc_fold{fidx + 1}_comparison.svg")
        out_path_png = os.path.join(out_dir, f"roc_fold{fidx + 1}_comparison.png")
        plt.close(fig)

        # PRC
        fig = plt.figure(figsize=(8, 5))
        ax = fig.gca()
        for method, fold_list, color, marker in zip(
            ["Baseline", "Ours"],
            [folds_b, folds_o],
            ["#54A0DD", "#E57067"],
            ['o', '^']
        ):
            if fidx >= len(fold_list):
                continue
            fold = fold_list[fidx]
            rec = np.array(fold.get("rec", []))
            prec = np.array(fold.get("prec", []))
            auprc = fold.get("auprc", float("nan"))
            if rec.size > 1:
                plt.plot(
                    rec, prec, label=f"{method} (AUPRC={auprc:.3f})", lw=2, color=color
                )
                plt.scatter(rec, prec, color=color, marker=marker, s=33, edgecolor="black", alpha=0.8, zorder=10, label=None)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"PRC Curve — Fold {fidx + 1}")
        handles, labels = ax.get_legend_handles_labels()
        out_handles = []; out_labels = []; seen = set()
        for h, l in zip(handles, labels):
            if l not in seen:
                seen.add(l)
                out_handles.append(h)
                out_labels.append(l)
        plt.legend(out_handles, out_labels)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        out_path_svg = os.path.join(out_dir, f"prc_fold{fidx + 1}_comparison.svg")
        out_path_png = os.path.join(out_dir, f"prc_fold{fidx + 1}_comparison.png")
        plt.close(fig)

def plot_from_json(json_path, out_dir=None):
    results = load_cv_results(json_path)
    if out_dir is None:
        out_dir = os.path.dirname(json_path) or "."
    print_fold_means(results)
    print_ranksum_tests(results)
    plot_barplot_metrics(results, out_dir)  # Only AUROC and AUPRC
    plot_roc_pr_per_fold(results, out_dir)
    print(f"Saved plots to: {out_dir}")

import os 

os.makedirs('cv5_outputs_all', exist_ok=True)
os.makedirs('cv5_outputs_all/task_1', exist_ok=True)
os.makedirs('cv5_outputs_all/task_2', exist_ok=True)

# -------
# 
# --------- USAGE ----------------
json_path = "cv5_outputs_all/task_1/cv5_results.json"
plot_from_json(json_path)

json_path = "cv5_outputs_all/task_2/cv5_results.json"
plot_from_json(json_path)
